# 🇸🇬 Singapore Smart City — Data Collector

Collects traffic images + metadata from all 90 LTA cameras every 60 seconds.
Data persists to Google Drive.

**Run time:** 6-12 hours per session | **Data rate:** ~50MB/hour | **GPU:** Not needed

In [ ]:
# 1. Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo & install dependencies
!git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git 2>/dev/null || (cd sg-smart-city-analytics && git pull)
!pip install -q aiohttp Pillow imagehash pyyaml click pandas

In [ ]:
# 3. Quick API health check
import requests
apis = {
    'Traffic Images': 'https://api.data.gov.sg/v1/transport/traffic-images',
    'Weather': 'https://api.data.gov.sg/v1/environment/24-hour-weather-forecast',
    'Temperature': 'https://api.data.gov.sg/v1/environment/air-temperature',
    'PM2.5': 'https://api.data.gov.sg/v1/environment/pm25',
    'Taxi': 'https://api.data.gov.sg/v1/transport/taxi-availability',
}
for name, url in apis.items():
    r = requests.get(url, timeout=10)
    status = '✅' if r.status_code == 200 else '❌'
    print(f'{status} {name}: HTTP {r.status_code}')

# Camera count
data = requests.get(apis['Traffic Images']).json()
cams = data['items'][0]['cameras']
print(f'\n📷 {len(cams)} cameras available')

In [ ]:
# 4. Configure collection
import os
os.chdir('/content/sg-smart-city-analytics')

DURATION_HOURS = 8       # Safe duration to avoid ungraceful Colab kills
INTERVAL_SECONDS = 60     # Seconds between cycles
DRIVE_OUTPUT = '/content/drive/MyDrive/sg_smart_city/data/raw'

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

estimated_cycles = int(DURATION_HOURS * 3600 / INTERVAL_SECONDS)
estimated_mb = DURATION_HOURS * 50
print(f'Duration:   {DURATION_HOURS} hours')
print(f'Interval:   {INTERVAL_SECONDS}s')
print(f'Cycles:     ~{estimated_cycles}')
print(f'Est. size:  ~{estimated_mb} MB')
print(f'Output:     {DRIVE_OUTPUT}')

### ⚠️ Anti-Disconnect Tip for Google Colab
Colab will disconnect if you leave the tab idle. To prevent this, open your browser's Developer Tools (F12) -> Console, and paste this:
```javascript
function keepAlive() {
    let btn = document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
    if(btn) { btn.click(); console.log("Clicked reconnect"); }
}
setInterval(keepAlive, 60000);
```

In [ ]:
# 4a. Check Google Drive Space
import shutil
total, used, free = shutil.disk_usage('/content/drive')
free_gb = free // (2**30)
print(f'Free space in Drive: {free_gb} GB')
if free_gb < 2:
    print('⚠️ WARNING: Less than 2GB free. Data collection might fail due to DiskQuotaExceeded.')


In [ ]:
# 5. Run continuous data collection
# This cell runs for DURATION_HOURS — don't interrupt it!
import asyncio
import aiohttp
import sys
import time
import json
import logging

sys.path.insert(0, '/content/sg-smart-city-analytics')
from src.ingestion.collector import DataCollector, SingaporeAPIClient, load_config

config = load_config('configs/collection_config.yaml')
config['collection']['output_dir'] = DRIVE_OUTPUT
config['collection']['interval_seconds'] = INTERVAL_SECONDS

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
logger = logging.getLogger('collector')

async def collect_continuous():
    collector = DataCollector(config)
    end_time = time.time() + (DURATION_HOURS * 3600)
    cycle = 0
    total_cameras = 0
    total_failures = 0

    async with aiohttp.ClientSession() as session:
        client = SingaporeAPIClient(config, session)

        while time.time() < end_time:
            cycle += 1
            cycle_start = time.time()

            try:
                result = await collector.run_collection_cycle(client)
                cameras_ok = result.get('cameras_success', 0)
                cameras_fail = result.get('cameras_failed', 0)
                total_cameras += cameras_ok
                total_failures += cameras_fail
                elapsed = time.time() - cycle_start

                hours_left = (end_time - time.time()) / 3600
                print(f'Cycle {cycle} | {cameras_ok}/90 cameras | '
                      f'{elapsed:.1f}s | {result.get("weather","?")} | '
                      f'{result.get("temperature","?")}°C | '
                      f'Total: {total_cameras} imgs | '
                      f'{hours_left:.1f}h remaining')
            except Exception as e:
                print(f'Cycle {cycle} | ERROR: {e}')

            # Sleep until next cycle
            sleep_time = max(0, INTERVAL_SECONDS - (time.time() - cycle_start))
            if time.time() + sleep_time < end_time:
                await asyncio.sleep(sleep_time)
            else:
                break

    print(f'\n{"="*50}')
    print(f'Collection complete!')
    print(f'Total cycles: {cycle}')
    print(f'Total images: {total_cameras}')
    print(f'Total failures: {total_failures}')
    print(f'Success rate: {total_cameras/(total_cameras+total_failures)*100:.1f}%' if total_cameras+total_failures > 0 else '')
    print(f'Data saved to: {DRIVE_OUTPUT}')

await collect_continuous()

In [ ]:
# 6. (After collection) Verify what we collected
import os
from pathlib import Path

data_path = Path(DRIVE_OUTPUT)
jpg_count = len(list(data_path.rglob('*.jpg')))
jsonl_count = len(list(data_path.rglob('*.jsonl')))
total_bytes = sum(f.stat().st_size for f in data_path.rglob('*') if f.is_file())
total_mb = total_bytes / (1024 * 1024)

date_dirs = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
camera_dirs = set()
for dd in data_path.iterdir():
    if dd.is_dir():
        for cd in dd.iterdir():
            if cd.is_dir():
                camera_dirs.add(cd.name)

print(f'📊 Collection Summary')
print(f'   Images:      {jpg_count:,}')
print(f'   Metadata:    {jsonl_count:,} files')
print(f'   Total size:  {total_mb:.1f} MB')
print(f'   Date range:  {date_dirs[0] if date_dirs else "N/A"} → {date_dirs[-1] if date_dirs else "N/A"}')
print(f'   Cameras:     {len(camera_dirs)} unique')